In [1]:
!pip install -qU google-genai langchain-google-genai langchain langchain-community langchain-text-splitters chromadb pypdf

In [2]:
import os
import getpass
import sys

In [3]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_50310/2726881704.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


In [4]:
def setup_env():
    if not os.getenv("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass(
            "Enter your Google API Key: "
        )

In [6]:
def load_and_split(filepath):

    if filepath.lower().endswith(".pdf"):
        loader = PyPDFLoader(filepath)

    elif filepath.lower().endswith(".txt"):
        loader = TextLoader(filepath, encoding="utf-8")

    else:
        raise ValueError("Please upload only PDF or TXT files.")

    docs = loader.load()

    print("Splitting document into chunks...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    splits = splitter.split_documents(docs)

    print(f"Created {len(splits)} chunks successfully!")

    return splits

In [7]:
def create_retriever(splits):

    print("Creating embeddings and vector store...")

    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        task_type="retrieval_document"
    )

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 6}
    )

    print("Retriever created successfully!")

    return retriever

In [9]:
from google.colab import files

setup_env()

uploaded = files.upload()

file_path = next(iter(uploaded))

print("Uploaded file:", file_path)

splits = load_and_split(file_path)

retriever = create_retriever(splits)

print("Your document is ready! You can ask questions now.")

Enter your Google API Key: ··········


StopIteration: 

In [10]:
from google import genai

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [11]:
while True:

    question = input("\nAsk Question (type exit to stop): ")

    if question.lower() in ["exit", "stop"]:
        print("Chat ended!")
        break

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    response = client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=f"""
        Answer the question using only the given context.

For counting questions, count the items mentioned
in the context carefully. You can calculate totals
from the available information.

Do not say information is missing if the answer
can be calculated from the context.

        If the answer is not available in the context,
        say: I could not find this information
        in the provided document.

        Context:
        {context}

        Question: {question}
        """
    )

    print("\nAnswer:", response.output_text)


Ask Question (type exit to stop): exit
Chat ended!
